# <font color="#418FDE" size="6.5" uppercase>**Text als Merkmale**</font>

>Last update: 20260823.
    
By the end of this Lecture, you will be able to:
- Strukturieren und bereinigen kleine lokale Textsammlungen mit Labels. 
- Erzeugen Textmerkmale mit Count-, TF-IDF- und Hashing-Vektorisierung. 
- Trainieren, validieren, analysieren und speichern einfache Textklassifikationspipelines. 


## **1. Texte vorbereiten**

### **1.1. Lokale Textsammlung**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_01_01.jpg?v=1787537889" width="250">



>* Lokale Textsammlungen sind klar abgegrenzte Datenbestände
>* Texteinheiten müssen konsistent festgelegt werden

>* Klare Labels ordnen Texte verlässlich ein
>* Einheitliche Kategorien verhindern verzerrte Analysen

>* Kleine Sammlungen lassen sich gut prüfen
>* Struktur, Qualität und Datenschutz früh klären



In [ ]:
#@title Python-Code - Lokale Textsammlung

# Wir bauen eine kleine lokale Textsammlung.
# Labels werden geprüft und vereinheitlicht.
# Die Ausgabe zeigt saubere Beobachtungen.

import pandas as pd

# Jeder Eintrag steht für eine lokale Textdatei.
raw_texts = [
    ["review_01.txt", "Positiv", "Tolles Produkt, schnelle Lieferung!"],
    ["review_02.txt", "negativ", "Leider defekt und sehr spät angekommen."],
    ["review_03.txt", "positiv ", "Gute Qualität, ich bestelle wieder."],
    ["review_04.txt", "Beschwerde", ""],
]

# Wir legen die Sammlung als übersichtliche Tabelle an.
texts = pd.DataFrame(raw_texts, columns=["file_name", "label", "text"])

# Labels werden klein geschrieben und Leerzeichen entfernt.
texts["clean_label"] = texts["label"].str.strip().str.lower()

# Leere Texte werden markiert, statt still übersehen zu werden.
texts["has_text"] = texts["text"].str.strip().str.len() > 0

# Nur bekannte Labels sollen später für Modelle verwendet werden.
allowed_labels = {"positiv", "negativ", "beschwerde"}
texts["label_ok"] = texts["clean_label"].isin(allowed_labels)

# Diese Prüfung macht typische Datenprobleme sofort sichtbar.
if len(texts) != 4:
    raise ValueError("Die Beispielsammlung sollte vier Texte enthalten.")

# Für die Modellierung behalten wir nur gültige Beobachtungen.
clean_texts = texts[texts["has_text"] & texts["label_ok"]].copy()

# Die Ausgabe bleibt klein und zeigt die wichtigste Struktur.
print("Lokale Textsammlung nach einfacher Bereinigung:")
print(clean_texts[["file_name", "clean_label", "text"]].to_string(index=False))
print("Verwendbare Texte:", len(clean_texts), "von", len(texts))



### **1.2. Texte bereinigen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_01_02.jpg?v=1787537893" width="250">



>* Uneinheitliche Textformen vor Analysen vereinheitlichen
>* Bereinigung schafft verlässliche Merkmale und Labels

>* Bereinigung immer am Analyseziel ausrichten
>* Bedeutungstragende Zeichen und Wörter bewahren

>* Labels und Dokumentzuordnung zuverlässig erhalten
>* Bereinigung dokumentieren und Ergebnisse überprüfbar machen



In [ ]:
#@title Python-Code - Texte bereinigen

# Wir bereinigen kurze Texte mit festen Labels.
# Der Fokus liegt auf nachvollziehbaren Textänderungen.
# Am Ende bleiben Text und Label verbunden.

import pandas as pd

# Diese kleine Sammlung enthält typische Rohtext-Probleme.
raw_data = pd.DataFrame(
    {
        "label": ["positiv", "negativ", "anfrage", "negativ"],
        "text": [
            "  Lieferung war SUPER!!! <br> Danke :)  ",
            "Produkt defekt...\nBitte schnell ersetzen!!!",
            "Hallo, gibt es Größe XL?   Viele Grüße",
            "NICHT zufrieden: Akku hält nur 2 Std. :(  ",
        ],
    }
)

# Eine Kopie schützt die ursprünglichen Rohtexte.
clean_data = raw_data.copy()

# Einfache Ersetzungen entfernen technische Artefakte.
clean_data["clean_text"] = clean_data["text"].str.replace("<br>", " ", regex=False)
clean_data["clean_text"] = clean_data["clean_text"].str.replace("\n", " ", regex=False)

# Kleinschreibung und Leerzeichen machen Schreibweisen einheitlicher.
clean_data["clean_text"] = clean_data["clean_text"].str.lower()
clean_data["clean_text"] = clean_data["clean_text"].str.split().str.join(" ")

# Diese Prüfung zeigt, dass kein Label verloren ging.
same_labels = clean_data["label"].equals(raw_data["label"])

# Eine kurze Tabelle vergleicht Rohtext und bereinigten Text.
preview = clean_data[["label", "text", "clean_text"]].head(4)
print("Labels unverändert:", same_labels)
print(preview.to_string(index=False))



### **1.3. Wörter zählen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_01_03.jpg?v=1787537891" width="250">



>* Texte in zählbare Wörter zerlegen
>* Gemeinsame Merkmale und erste Muster erkennen

>* Vorher festlegen, was als Wort zählt
>* Relevante Wörter hängen von Aufgabe ab

>* Zählergebnisse kritisch auf Fehler prüfen
>* Themen, Verzerrungen und Gruppenunterschiede erkennen



In [ ]:
#@title Python-Code - Wörter zählen

# Dieses Beispiel zählt Wörter in kurzen Texten.
# Labels helfen beim Vergleichen kleiner Textsammlungen.
# Die Ausgabe zeigt häufige bereinigte Wörter.

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# Eine kleine lokale Sammlung enthält Texte und Labels.
texts = [
    "Passwort vergessen, bitte Zugang zurücksetzen.",
    "Die Rechnung enthält einen falschen Betrag.",
    "Passwort funktioniert nach dem Update nicht.",
    "Bitte senden Sie die Rechnung erneut.",
]

labels = ["Zugang", "Rechnung", "Zugang", "Rechnung"]

# CountVectorizer bereinigt einfache Wörter und zählt sie.
vectorizer = CountVectorizer(lowercase=True)
word_matrix = vectorizer.fit_transform(texts)

# Die Matrix wird als kleine Tabelle lesbar gemacht.
word_table = pd.DataFrame(
    word_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
)

if word_table.shape[0] != len(texts):
    raise ValueError("Die Anzahl der Textzeilen passt nicht.")

# Wir ergänzen Labels, um Unterschiede sichtbar zu machen.
word_table.insert(0, "label", labels)

# Summen zeigen, welche Wörter insgesamt häufig sind.
word_counts = word_table.drop(columns="label").sum().sort_values(ascending=False)
top_words = word_counts.head(5)

print("Wortzählung für vier kurze Texte:")
print(word_table[["label", "passwort", "rechnung", "bitte", "die"]])
print("Häufigste Wörter insgesamt:")
print(top_words.to_string())



## **2. Texte vektorisieren**

### **2.1. n Gramme und Stopwörter**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_02_01.jpg?v=1787537897" width="250">



>* n-Gramme machen Texte zu nutzbaren Merkmalen
>* Bigramme erfassen wichtigen lokalen Kontext

>* Unigramme zeigen robuste Schlüsselwort-Hinweise.
>* Größere n-Gramme erfassen Phrasen, erhöhen Komplexität.

>* Stopwörter entfernen kann Merkmale reduzieren
>* Bedeutung und Validierung vorher prüfen



In [ ]:
#@title Python-Code - n Gramme und Stopwörter

# Dieses Beispiel zeigt n-Gramme und Stopwörter.
# CountVectorizer erzeugt sichtbare Textmerkmale aus kurzen Sätzen.
# Die Ausgabe vergleicht Merkmale mit und ohne Stopwörter.

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# Kleine Texte machen die Wirkung der Einstellungen gut sichtbar.
texts = [
    "der Film war nicht gut",
    "der Film war sehr gut",
    "das Essen war nicht schlecht",
    "das Essen war sehr schlecht",
]

# Diese Stopwörter entfernen häufige Funktionswörter, aber keine Verneinungen.
german_stop_words = ["der", "das", "war", "sehr"]

# Unigramme zählen einzelne Wörter als Merkmale.
unigram_vectorizer = CountVectorizer(ngram_range=(1, 1))
unigram_matrix = unigram_vectorizer.fit_transform(texts)

# Bigramme erfassen kurze Wortfolgen wie nicht gut.
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2))
bigram_matrix = bigram_vectorizer.fit_transform(texts)

# Stopwörter verkleinern den Merkmalsraum gezielt.
stop_vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    stop_words=german_stop_words,
)
stop_matrix = stop_vectorizer.fit_transform(texts)

# Eine einfache Prüfung schützt vor unerwartet leeren Merkmalen.
if stop_matrix.shape[1] == 0:
    raise ValueError("Die Stopwortliste hat alle Merkmale entfernt.")

# Die Tabelle zeigt, wie viele Merkmale jede Variante erzeugt.
summary = pd.DataFrame(
    {
        "Variante": ["Unigramme", "Unigramme und Bigramme", "Mit Stopwörtern entfernt"],
        "Merkmale": [unigram_matrix.shape[1], bigram_matrix.shape[1], stop_matrix.shape[1]],
    }
)

# Ausgewählte Merkmale zeigen den Unterschied besonders anschaulich.
selected_features = ["gut", "nicht", "nicht gut", "sehr gut"]
feature_names = bigram_vectorizer.get_feature_names_out()

# Fehlende Merkmale erhalten den Wert null.
first_text_counts = []
for feature in selected_features:
    if feature in feature_names:
        column_index = list(feature_names).index(feature)
        first_text_counts.append(int(bigram_matrix[0, column_index]))
    else:
        first_text_counts.append(0)

print("scikit-learn erzeugt hier Count-Merkmale aus Text.")
print(summary.to_string(index=False))
print("Erster Text: der Film war nicht gut")
print(dict(zip(selected_features, first_text_counts)))



### **2.2. TF IDF Gewichtung**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_02_02.jpg?v=1787537895" width="250">



>* TF-IDF bewertet Wörter nach Dokumenttypik
>* Seltene, aussagekräftige Begriffe zählen stärker

>* TF zählt Wörter im einzelnen Dokument
>* IDF schwächt allgemeine Wörter ab

>* TF-IDF ist effizient und gut interpretierbar
>* Begrenzt Bedeutung, strukturiert Text für Modelle



In [ ]:
#@title Python-Code - TF IDF Gewichtung

# Dieses Beispiel zeigt TF-IDF-Gewichte für kurze Texte.
# Häufige Wörter werden niedriger gewichtet als spezifische Wörter.
# Die Ausgabe vergleicht Count- und TF-IDF-Merkmale.

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import sklearn

# Eine kleine Textsammlung macht die Gewichtung gut sichtbar.
documents = [
    "akku akku laufzeit laden produkt",
    "display display bruch reparatur produkt",
    "akku laden kabel produkt",
    "rechnung vertrag kündigung produkt",
]

# CountVectorizer zählt nur, wie oft Wörter vorkommen.
count_vectorizer = CountVectorizer()
count_matrix = count_vectorizer.fit_transform(documents)

# TfidfVectorizer gewichtet Wörter nach Häufigkeit und Seltenheit.
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

# Die Merkmalsnamen müssen in beiden Darstellungen gleich sein.
feature_names = count_vectorizer.get_feature_names_out()
if list(feature_names) != list(tfidf_vectorizer.get_feature_names_out()):
    raise ValueError("Die Merkmalsnamen passen nicht zusammen.")

# Wir betrachten ein Dokument mit allgemeinen und spezifischen Wörtern.
row_index = 0
count_values = count_matrix[row_index].toarray()[0]
tfidf_values = tfidf_matrix[row_index].toarray()[0]

# Eine kleine Tabelle zeigt den Unterschied übersichtlich.
comparison = pd.DataFrame(
    {"wort": feature_names, "count": count_values, "tfidf": tfidf_values}
)

# Nur Wörter aus dem ersten Dokument sind für den Vergleich nötig.
comparison = comparison[comparison["count"] > 0].copy()
comparison["tfidf"] = comparison["tfidf"].round(3)
comparison = comparison.sort_values("tfidf", ascending=False)

print("scikit-learn Version:", sklearn.__version__)
print("Dokument 1:", documents[row_index])
print(comparison.to_string(index=False))



### **2.3. Hashing und Sparse**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_02_03.jpg?v=1787537898" width="250">



>* Text erzeugt sehr viele mögliche Merkmale
>* Hashing ordnet Tokens festen Spalten zu

>* Feste Merkmalszahl spart Speicher und Rechenzeit
>* Kollisionen erschweren eindeutige Interpretation

>* Sparse-Matrizen speichern nur vorhandene Textmerkmale
>* Hashing bleibt dadurch speichereffizient nutzbar



In [ ]:
#@title Python-Code - Hashing und Sparse

# Dieses Beispiel zeigt Hashing-Vektoren für kurze Texte.
# Sparse-Matrizen speichern nur vorhandene Textmerkmale effizient.
# Die Ausgabe vergleicht Dimensionen, Nichtnullwerte und Kollisionen.

import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import HashingVectorizer
import sklearn

# Eine kleine Textsammlung reicht für das Grundprinzip.
texts = [
    "gutes zimmer gutes fruehstueck",
    "schlechtes zimmer laute strasse",
    "gutes personal ruhige lage",
    "laute musik schlechtes fruehstueck",
]

# CountVectorizer baut ein sichtbares Vokabular auf.
count_vectorizer = CountVectorizer()
count_matrix = count_vectorizer.fit_transform(texts)

# HashingVectorizer nutzt feste Spalten ohne gespeichertes Vokabular.
hash_vectorizer = HashingVectorizer(
    n_features=8,
    alternate_sign=False,
    norm=None,
)

hash_matrix = hash_vectorizer.transform(texts)

# Sparse-Matrizen speichern nur Werte ungleich null.
count_density = count_matrix.nnz / np.prod(count_matrix.shape)
hash_density = hash_matrix.nnz / np.prod(hash_matrix.shape)

# Kleine Hashing-Dimensionen machen Kollisionen gut sichtbar.
unique_words = sorted(count_vectorizer.vocabulary_)
hashed_columns = hash_vectorizer.transform(unique_words).nonzero()[1]

collision_count = len(unique_words) - len(set(hashed_columns))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Count-Form: {count_matrix.shape}, Nichtnullwerte: {count_matrix.nnz}")
print(f"Hashing-Form: {hash_matrix.shape}, Nichtnullwerte: {hash_matrix.nnz}")
print(f"Count-Dichte: {count_density:.2f}, Hashing-Dichte: {hash_density:.2f}")
print(f"Vokabulargröße: {len(unique_words)}, Hash-Kollisionen: {collision_count}")
print(f"Beispielspalten: {list(zip(unique_words[:5], hashed_columns[:5]))}")



## **3. Textpipeline trainieren**

### **3.1. Naive Bayes Klassifikation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_03_01.jpg?v=1787537900" width="250">



>* Lernt typische Wortmuster aus gelabelten Texten
>* Einfach, schnell und robust für Textprojekte

>* Text zuerst in Zahlenmerkmale umwandeln
>* Gute Daten und Vorverarbeitung entscheiden

>* Fehler und Klassenverzerrungen gezielt untersuchen
>* Gespeicherte Pipelines sichern gleiche Textverarbeitung



In [ ]:
#@title Python-Code - Naive Bayes Klassifikation

# Wir trainieren eine kleine Textpipeline.
# Naive Bayes lernt aus Wortmerkmalen.
# Die Ausgabe zeigt Bewertung und Vorhersagen.

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Diese Mini-Daten simulieren gelabelte Kundenrückmeldungen.
texts = [
    "defekt rückerstattung funktioniert nicht",
    "gerät kaputt bitte ersatz",
    "lieferung beschädigt und verspätet",
    "rechnung falsch bitte korrigieren",
    "danke sehr hilfreich und freundlich",
    "super service ich bin zufrieden",
    "schnelle antwort vielen dank",
    "produkt gefällt mir sehr gut",
    "wann kommt meine bestellung",
    "wie ändere ich meine adresse",
    "frage zur garantie und reparatur",
    "kann ich den termin verschieben",
]

# Die Labels sind die Zielklassen für das Modell.
labels = [
    "Beschwerde",
    "Beschwerde",
    "Beschwerde",
    "Beschwerde",
    "Lob",
    "Lob",
    "Lob",
    "Lob",
    "Frage",
    "Frage",
    "Frage",
    "Frage",
]

# Eine einfache Prüfung verhindert unpassende Eingabelängen.
if len(texts) != len(labels):
    raise ValueError("Texte und Labels müssen gleich lang sein.")

# Die Aufteilung trennt Training und Test fair voneinander.
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)

# Die Pipeline vektorisiert Text und trainiert Naive Bayes.
pipeline = Pipeline(
    [("tfidf", TfidfVectorizer()), ("model", MultinomialNB())]
)

# Nur Trainingsdaten werden zum Lernen verwendet.
pipeline.fit(train_texts, train_labels)

# Der Testdatensatz prüft unbekannte Beispiele.
predicted_labels = pipeline.predict(test_texts)
accuracy = accuracy_score(test_labels, predicted_labels)

# Neue Texte laufen durch dieselbe gespeicherte Pipeline-Struktur.
new_texts = [
    "bitte rückerstattung das produkt ist kaputt",
    "vielen dank für den schnellen service",
    "wie kann ich meine bestellung ändern",
]
new_predictions = pipeline.predict(new_texts)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Testgenauigkeit: {accuracy:.2f}")
print(f"Testbeispiele: {len(test_texts)}")
print(f"Neu 1: {new_predictions[0]}")
print(f"Neu 2: {new_predictions[1]}")
print(f"Neu 3: {new_predictions[2]}")



### **3.2. Modelle vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_03_02.jpg?v=1787537904" width="250">



>* Textpipelines systematisch und fair vergleichen
>* Merkmale und Modelle passend auswählen

>* Trainings- und Validierungsdaten strikt trennen
>* Klassenfehler differenziert und fachlich bewerten

>* Modelle fachlich und praktisch bewerten
>* Gewählte Pipeline speichern und reproduzierbar nutzen



In [ ]:
#@title Python-Code - Modelle vergleichen

# Wir vergleichen einfache Textpipelines fair.
# Gleiche Daten machen Modellunterschiede sichtbar.
# Die beste Pipeline wird klar benannt.

import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Diese kleine Textsammlung hat drei klare Klassen.
texts = [
    "rechnung betrag zahlung mahnung",
    "rechnung falsch betrag konto",
    "zahlung offen rechnung bitte",
    "mahnung zahlung konto betrag",
    "drucker fehler wlan verbindung",
    "internet langsam router fehler",
    "software absturz update fehler",
    "wlan router verbindung problem",
    "vertrag kündigen frist bestätigen",
    "kündigung abo vertrag sofort",
    "abo beenden bestätigung kündigung",
    "frist vertrag kündigen bitte",
]

# Die Labels beschreiben das gewünschte Thema.
labels = [
    "Rechnung",
    "Rechnung",
    "Rechnung",
    "Rechnung",
    "Technik",
    "Technik",
    "Technik",
    "Technik",
    "Kündigung",
    "Kündigung",
    "Kündigung",
    "Kündigung",
]

# Eine einfache Prüfung verhindert unklare Trainingsdaten.
if len(texts) != len(labels):
    raise ValueError("Texte und Labels müssen gleich lang sein.")

# Stratifikation hält die Klassenverteilung im Vergleich stabil.
train_texts, valid_texts, train_labels, valid_labels = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)

# Jede Pipeline enthält Vektorisierung und Klassifikationsmodell.
pipelines = {
    "Count + Naive Bayes": Pipeline(
        [("vectorizer", CountVectorizer()), ("model", MultinomialNB())]
    ),
    "TF-IDF + Logistische Regression": Pipeline(
        [("vectorizer", TfidfVectorizer()), ("model", LogisticRegression(max_iter=200))]
    ),
}

# Alle Varianten sehen exakt dieselben Trainingsdaten.
results = []
for name, pipeline in pipelines.items():
    pipeline.fit(train_texts, train_labels)
    predictions = pipeline.predict(valid_texts)
    score = accuracy_score(valid_labels, predictions)
    results.append((name, score))

# Die Ausgabe bleibt kurz und vergleichbar.
print(f"scikit-learn Version: {sklearn.__version__}")
for name, score in results:
    print(f"{name}: Validierungsgenauigkeit {score:.2f}")

# Die höchste Validierungsgenauigkeit bestimmt hier die Auswahl.
best_name, best_score = max(results, key=lambda item: item[1])
print(f"Ausgewählte Pipeline: {best_name} ({best_score:.2f})")



### **3.3. Eigenes Textprojekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_B/image_03_03.jpg?v=1787537902" width="250">



>* Kleine Textsammlung und klare Frage wählen
>* Kategorien fachlich sinnvoll und eindeutig definieren

>* Pipeline wiederholbar trainieren und validieren
>* Fehler zeigen Daten- und Labelprobleme

>* Gespeicherte Pipelines sichern wiederholbare Textverarbeitung
>* Modellgrenzen kritisch und verantwortungsvoll einschätzen



In [ ]:
#@title Python-Code - Eigenes Textprojekt

# Dieses Beispiel trainiert eine kleine Textpipeline.
# TF-IDF wandelt Texte in Merkmale um.
# Die Ausgabe bewertet und nutzt das Modell.

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Kleine lokale Texte ersetzen hier eine externe Datei.
texts = [
    "Die Lieferung war schnell und das Produkt ist sehr gut",
    "Ich bin zufrieden und würde wieder bestellen",
    "Tolle Qualität und freundlicher Service",
    "Das Gerät funktioniert zuverlässig und macht Freude",
    "Sehr schlechte Verpackung und der Artikel war kaputt",
    "Ich bin enttäuscht und möchte mein Geld zurück",
    "Der Support antwortet nicht und das Problem bleibt",
    "Die Bestellung kam zu spät und war beschädigt",
    "Wann wird meine Rechnung für die Bestellung erstellt",
    "Ich brauche Hilfe beim Ändern meiner Lieferadresse",
    "Bitte senden Sie mir Informationen zum Rückgabeprozess",
    "Wie kann ich den Termin für die Lieferung verschieben",
]

# Die Labels beschreiben die gewünschte Klassifikationsfrage.
labels = [
    "positiv",
    "positiv",
    "positiv",
    "positiv",
    "negativ",
    "negativ",
    "negativ",
    "negativ",
    "frage",
    "frage",
    "frage",
    "frage",
]

# Eine einfache Prüfung schützt vor falsch ausgerichteten Daten.
if len(texts) != len(labels):
    raise ValueError("Jeder Text braucht genau ein Label.")

# Der Split hält unbekannte Beispiele für die Validierung zurück.
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    labels,
    test_size=0.33,
    random_state=42,
    stratify=labels,
)

# Die Pipeline verhindert unterschiedliche Vorbereitungsschritte.
pipeline = Pipeline(
    [
        ("vectorizer", TfidfVectorizer(lowercase=True)),
        ("model", LogisticRegression(max_iter=200, random_state=42)),
    ]
)

# Nur Trainingsdaten werden zum Lernen verwendet.
pipeline.fit(train_texts, train_labels)

# Die zurückgehaltenen Texte prüfen die Generalisierung.
predicted_labels = pipeline.predict(test_texts)
accuracy = accuracy_score(test_labels, predicted_labels)

# Neue Texte laufen durch dieselbe gespeicherte Pipeline-Struktur.
new_texts = [
    "Der Artikel ist kaputt und ich bin enttäuscht",
    "Bitte ändern Sie meine Lieferadresse",
]
new_predictions = pipeline.predict(new_texts)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Validierungsgenauigkeit: {accuracy:.2f}")
print(f"Testbeispiele: {len(test_texts)}")
print(f"Neuer Text 1: {new_predictions[0]}")
print(f"Neuer Text 2: {new_predictions[1]}")



# <font color="#418FDE" size="6.5" uppercase>**Text als Merkmale**</font>


In this lecture, you learned to:
- Strukturieren und bereinigen kleine lokale Textsammlungen mit Labels. 
- Erzeugen Textmerkmale mit Count-, TF-IDF- und Hashing-Vektorisierung. 
- Trainieren, validieren, analysieren und speichern einfache Textklassifikationspipelines. 

In the next Module (Module 13), we will go over 'Bild- und Signalmodelle'